# 🎬 Netflix Movie Recommendation System
### ML Project Notebook

## **1. Project Overview**

### 📌 Project Title

**Netflix-Style Movie Recommendation System using Machine Learning**

---

### 🎯 Objective

The objective of this project is to build an intelligent movie recommendation system that suggests similar movies to users based on their preferences using machine learning techniques.

---

### ❓ Problem Statement

With the rapid growth of OTT platforms, users face difficulty in discovering relevant content due to the vast number of available movies. This project aims to solve the problem by providing personalized movie recommendations using content-based filtering.

---

## **2. Data Description**

### 📊 Data Source

The dataset was obtained from **TMDB (The Movie Database)** and Kaggle, containing metadata about movies.

---

### 📦 Data Volume

* ~5000 movies
* Features include:

  * Title
  * Overview
  * Genres
  * Keywords

---

### 🔑 Key Variables

* `original_title` → Movie name
* `overview` → Description of movie
* `genres` → Category (Action, Drama, etc.)
* `keywords` → Important tags

---

### 🧹 Data Cleaning & Preprocessing

* Removed null values
* Selected relevant columns
* Combined features into a single **“tags” column**
* Converted text to lowercase
* Removed stopwords using TF-IDF

---

## **3. Target Audience**

This system is useful for:

* OTT platforms (like Netflix, Prime Video)
* Movie recommendation engines
* Users looking for personalized suggestions

---

## **4. Machine Learning Workflow**

### 🤖 Model Selection

This project uses a **Content-Based Filtering approach** instead of traditional classification.

---

### 🧠 Features & Techniques

* **Feature Engineering:**

  * Combined `overview + genres + keywords`

* **Vectorization:**

  * Used **TF-IDF (Term Frequency - Inverse Document Frequency)**
  * Converts text into numerical vectors

* **Similarity Measurement:**

  * Used **Cosine Similarity** to find similar movies

---

### ✂️ Train-Test Split

❌ Not required
✔ Because this is a **recommendation system**, not a prediction model

---

## **5. Model Evaluation**

### 📏 Metrics Used

Instead of accuracy, evaluation is based on:

* Relevance of recommendations
* Similarity score between movies

---

### 📈 Results

* System successfully recommends **top 5–10 similar movies**
* Handles real-world scenarios using **fuzzy matching**
* Integrated with TMDB API for:

  * Posters
  * Ratings
  * Trending movies

---

## **6. Project Scope and Limitations**

### ✅ Scope

* Personalized movie recommendation
* Real-time data using API
* Netflix-style UI using Streamlit
* Watchlist functionality

---

### ⚠️ Limitations

* Does not consider user behavior (no collaborative filtering)
* Depends on dataset quality
* Cold-start problem (new movies/users)

---

## **7. Outcome / Expected Results**

### 🎯 Final Outcome

* Users can:

  * Search movies
  * View details (poster, rating, overview)
  * Get similar recommendations
  * Add movies to watchlist

---

### 💡 Business Value

* Improves user engagement
* Enhances content discovery
* Can be integrated into OTT platforms

---


> “This project implements a hybrid recommendation system combining content-based filtering using TF-IDF and cosine similarity with real-time TMDB API integration, providing a scalable and user-friendly Netflix-like experience.”

---

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches
import pickle
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load Dataset

In [4]:
movies = pd.read_csv('dataset.csv')
movies.head()

,id,title,genre,original_language,overview,popularity,release_date,vote_average,vote_count
0,278,The Shawshank Redemption,"Drama,Crime",en,Framed in the 1940s for the double murder of h...,94.075,1994-09-23,8.7,21862
1,19404,Dilwale Dulhania Le Jayenge,"Comedy,Drama,Romance",hi,"Raj is a rich, carefree, happy-go-lucky second...",25.408,1995-10-19,8.7,3731
2,238,The Godfather,"Drama,Crime",en,"Spanning the years 1945 to 1955, a chronicle o...",90.585,1972-03-14,8.7,16280
3,424,Schindler's List,"Drama,History,War",en,The true story of how businessman Oskar Schind...,44.761,1993-12-15,8.6,12959
4,240,The Godfather: Part II,"Drama,Crime",en,In the continuing saga of the Corleone crime f...,57.749,1974-12-20,8.6,9811


## Step 3: Data Exploration

In [5]:
movies.info()
movies.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10000 non-null  int64  
 1   title              10000 non-null  object 
 2   genre              9997 non-null   object 
 3   original_language  10000 non-null  object 
 4   overview           9987 non-null   object 
 5   popularity         10000 non-null  float64
 6   release_date       10000 non-null  object 
 7   vote_average       10000 non-null  float64
 8   vote_count         10000 non-null  int64  
dtypes: float64(2), int64(2), object(5)
memory usage: 703.3+ KB


id                    0
title                 0
genre                 3
original_language     0
overview             13
popularity            0
release_date          0
vote_average          0
vote_count            0
dtype: int64

## Step 4: Feature Selection

In [6]:
movies = movies[['id', 'title', 'overview', 'genre']]
movies.head()

,id,title,overview,genre
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,"Drama,Crime"
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance"
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime"
3,424,Schindler's List,The true story of how businessman Oskar Schind...,"Drama,History,War"
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...,"Drama,Crime"


## Step 5: Feature Engineering

In [7]:
movies['tags'] = movies['overview'] + " " + movies['genre']
movies.head()

,id,title,overview,genre,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...,"Drama,Crime",Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second...","Comedy,Drama,Romance","Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama,Crime","Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...,"Drama,History,War",The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...,"Drama,Crime",In the continuing saga of the Corleone crime f...


## Step 6: Drop Unused Columns

In [8]:
new_data = movies.drop(columns=['overview', 'genre'])
new_data.head()

,id,title,tags
0,278,The Shawshank Redemption,Framed in the 1940s for the double murder of h...
1,19404,Dilwale Dulhania Le Jayenge,"Raj is a rich, carefree, happy-go-lucky second..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o..."
3,424,Schindler's List,The true story of how businessman Oskar Schind...
4,240,The Godfather: Part II,In the continuing saga of the Corleone crime f...


## Step 7: Text Vectorization

In [9]:
cv = CountVectorizer(max_features=10000, stop_words='english')
vector = cv.fit_transform(new_data['tags'].astype('U')).toarray()
vector.shape

(10000, 10000)

## Step 8: Cosine Similarity

In [10]:
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.05634362, 0.12888482, ..., 0.07559289, 0.11065667,
        0.06900656],
       [0.05634362, 1.        , 0.07624929, ..., 0.        , 0.03636965,
        0.        ],
       [0.12888482, 0.07624929, 1.        , ..., 0.02273314, 0.06655583,
        0.09338592],
       ...,
       [0.07559289, 0.        , 0.02273314, ..., 1.        , 0.03253   ,
        0.03042903],
       [0.11065667, 0.03636965, 0.06655583, ..., 0.03253   , 1.        ,
        0.04454354],
       [0.06900656, 0.        , 0.09338592, ..., 0.03042903, 0.04454354,
        1.        ]], shape=(10000, 10000))

## Step 9: Improved Recommendation Function

In [11]:
def recommend(movie):
    titles = new_data['title'].tolist()
    match = get_close_matches(movie, titles, n=1, cutoff=0.6)

    if not match:
        print("Movie not found")
        return

    movie = match[0]

    index = new_data[new_data['title'] == movie].index[0]
    distances = similarity[index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movie_list:
        print(new_data.iloc[i[0]].title)

## Step 10: Test Recommendation

In [12]:
recommend("Iron Man")

Iron Man 3
Guardians of the Galaxy Vol. 2
Avengers: Age of Ultron
Star Wars: Episode III - Revenge of the Sith
Iron Man 2


## Step 11: Save Model

In [13]:
pickle.dump(new_data, open('movies_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))